In [ ]:
import os
import speech_recognition as sr
import pyttsx3
from dotenv import load_dotenv
from google import genai


load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env")


client = genai.Client(api_key=api_key)

recognizer = sr.Recognizer()
engine = pyttsx3.init()

recognizer.energy_threshold = 250
recognizer.dynamic_energy_threshold = True
recognizer.pause_threshold = 0.6
recognizer.phrase_threshold = 0.2
recognizer.non_speaking_duration = 0.3


def speak(text):
    print("Assistant:", text)
    engine.say(text)
    engine.runAndWait()


def listen():
    with sr.Microphone() as source:
        print("\nListening...")

        try:
            audio = recognizer.listen(
                source,
                timeout=1,
                phrase_time_limit=5
            )
        except sr.WaitTimeoutError:
            return ""
        except OSError as e:
            print("Microphone error:", e)
            return ""

    try:
        text = recognizer.recognize_google(
            audio,
            language="en-IN"
        )

        print("You:", text)
        return text.lower()

    except sr.UnknownValueError:
        print("Couldn't understand.")
        return ""

    except sr.RequestError as e:
        print("Speech recognition error:", e)
        speak("I can't connect to the speech recognition service.")
        return ""


def ask_gemini(text):
    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=text
        )

        return response.text

    except Exception as e:
        print("Gemini error:", e)
        return "Sorry, I couldn't process that."


def process_command(command):
    if command in ["stop", "exit", "quit", "goodbye"]:
        speak("Goodbye!")
        return False

    answer = ask_gemini(command)
    speak(answer)

    return True


print("Calibrating microphone...")

with sr.Microphone() as source:
    recognizer.adjust_for_ambient_noise(
        source,
        duration=1
    )

print("Microphone ready.")

speak("Hello! I am ready.")

while True:
    command = listen()

    if not command:
        continue

    if not process_command(command):
        break